In [1]:
import json
import requests
import time
from typing import List
from openai import OpenAI
import os

import asyncio
from typing import List
from tqdm import tqdm
from openai import AsyncOpenAI
import aiohttp
from concurrent.futures import ThreadPoolExecutor
import time
API_KEY = "sk-6faadffd5af649e8b535488b369e795d"
system_prompt = '''你是一个专注于AI和大语言模型领域的专业翻译助手。请基于以下要求执行翻译任务：

### 🔧 核心指令
- **任务类型**：零样本翻译(zero-shot translation)
- **领域**：大语言模型(LLM)、人工智能(AI)、机器学习(ML)、深度学习
- **风格**：技术文档风格，保持专业性和准确性

### 📚 术语处理规范
**必须保持原样的术语**：
- 模型名称：GPT、BERT、Transformer、ResNet、YOLO,token,prompt等
- 技术概念：注意力机制(attention mechanism)、微调(fine-tuning)、提示工程(prompt engineering)、思维链(chain-of-thought)
- 架构组件：编码器(encoder)、解码器(decoder)、嵌入层(embedding layer)、位置编码(positional encoding)
- 训练方法：预训练(pre-training)、指令调优(instruction tuning)、人类反馈强化学习(RLHF)、参数高效微调(PEFT)
- 评估指标：困惑度(perplexity)、BLEU分数、ROUGE分数、准确率(accuracy)

**需要翻译的术语**：
- 技术描述、论文摘要、方法章节、实验结果等叙述性内容
- 非专有名词的技术解释
- 专有名词可在后面用括号标注英文

### 🎯 翻译质量要求
1. **保持技术准确性**：确保所有技术概念的准确传达
2. **上下文一致性**：在同一文档中保持术语翻译的一致性
3. **格式保留**：保留所有代码块、数学公式、参考文献格式
4. **文化适应性**：适当调整表达方式以适应目标语言读者的理解习惯，可适当顺滑。

### ⚡ 处理策略
- **多义词消歧**：根据上下文确定多义词的正确翻译
- **长句分解**：对复杂的技术长句进行合理的句式重构
- **被动语态处理**：根据目标语言习惯调整被动语态的使用
- **技术缩略语**：首次出现时提供全称，后续可使用缩写

### 📋 输出格式
直接输出翻译结果，不添加额外的任何东西，只需要返回["翻译的中文"]

**当前翻译任务**：
文本语言：英语
目标语言：简体中文
'''

def batch_translate_subtitles(subtitles: List[str], target_language: str = "中文", batch_size: int = 20) -> List[str]:
    """
    批量翻译字幕，提高效率
    
    Args:
        subtitles: 字幕列表
        target_language: 目标语言
        batch_size: 每批处理的数量，根据字幕平均长度调整
    """
   
    client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com")
    
   
    translated_subtitles = []
    
    for batch_start in tqdm(range(0, len(subtitles), batch_size)):
        batch = subtitles[batch_start:batch_start + batch_size]
        # 构建一个包含所有批处理字幕的提示词
        batch_prompt = f"请将以下文本逐条翻译成{target_language}，直接返回翻译结果：{batch}\n"
        try:
            response = client.chat.completions.create(
                                                    model="deepseek-chat",
                                                    messages=[
                                                        {"role": "system", "content": system_prompt},
                                                        {"role": "user", "content":batch_prompt},
                                                    ],
                                                    stream=False,
                                                    temperature= 0,)
                                        
            full_translation = response.choices[0].message.content                                
            if full_translation:
            #print(response.choices[0].message.content)
                #full_translation = response.choices[0].message.content
                translated_subtitles.extend((full_translation,batch))
            else:
                print(f"批量翻译批次 {batch_start//batch_size + 1} 失败，状态码: {response}")
            
        except Exception as e:
            print(f"批量翻译批次 {batch_start//batch_size + 1} 异常: {e}")
            translated_subtitles.extend(batch)
        

    
    return translated_subtitles

# 同步多线程版本
def batch_translate_subtitles_threaded(
    subtitles: List[str], 
    target_language: str = "中文", 
    batch_size: int = 20,
    max_workers: int = 5
) -> List[str]:
    """
    多线程批量翻译字幕
    
    Args:
        subtitles: 字幕列表
        target_language: 目标语言
        batch_size: 每批处理的数量
        max_workers: 最大线程数
    """
    
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://api.deepseek.com"
    )
    
    def translate_single_batch(args):
        """翻译单个批次（用于线程池）"""
        batch, batch_num = args
        batch_prompt = f"请将以下文本逐条翻译成{target_language}，直接返回翻译结果：{batch}\n"
        
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": batch_prompt},
                ],
                stream=False,
                temperature=0,
            )
            
            full_translation = response.choices[0].message.content
            if full_translation:
                return full_translation
            else:
                print(f"批量翻译批次 {batch_num} 失败")
                return batch
                
        except Exception as e:
            print(f"批量翻译批次 {batch_num} 异常: {e}")
            return batch
    
    # 准备批次数据
    batch_args = []
    for batch_start in range(0, len(subtitles), batch_size):
        batch = subtitles[batch_start:batch_start + batch_size]
        batch_num = batch_start // batch_size + 1
        batch_args.append((batch, batch_num))
    
    # 使用线程池并发执行
    translated_subtitles = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(translate_single_batch, batch_args),
            total=len(batch_args),
            desc="翻译进度"
        ))
        translated_subtitles.extend(results)
    
    return translated_subtitles


# 异步版本
async def batch_translate_subtitles_async(
    subtitles: List[str], 
    target_language: str = "中文", 
    batch_size: int = 20,
    max_concurrent: int = 5
) -> List[str]:
    """
    异步批量翻译字幕，提高效率
    
    Args:
        subtitles: 字幕列表
        target_language: 目标语言
        batch_size: 每批处理的数量
        max_concurrent: 最大并发数
    """
    
    client = AsyncOpenAI(
        api_key=API_KEY,
        base_url="https://api.deepseek.com"
    )
    
    translated_subtitles = []
    
    async def translate_batch(batch: List[str], batch_num: int) -> List[str]:
        """翻译单个批次"""
        batch_prompt = f"请将以下文本逐条翻译成{target_language}，直接返回翻译结果：{batch}\n"
        
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": batch_prompt},
                ],
                stream=False,
                temperature=0,
            )
            
            full_translation = response.choices[0].message.content
            if full_translation:
                return full_translation
            else:
                print(f"批量翻译批次 {batch_num} 失败")
                return batch
                
        except Exception as e:
            print(f"批量翻译批次 {batch_num} 异常: {e}")
            return batch
    
    # 创建所有批次任务
    batches = []
    batch_numbers = []
    for batch_start in range(0, len(subtitles), batch_size):
        batch = subtitles[batch_start:batch_start + batch_size]
        batches.append(batch)
        batch_numbers.append(batch_start // batch_size + 1)
    
    # 使用信号量控制并发数
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def bounded_translate(batch, batch_num):
        async with semaphore:
            return await translate_batch(batch, batch_num)
    
    # 并发执行所有批次
    tasks = [bounded_translate(batch, num) for batch, num in zip(batches, batch_numbers)]
    
    # 使用tqdm显示进度
    results = []
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="翻译进度"):
        result = await task
        results.append(result)
    
    # 按原始顺序整理结果
    translated_subtitles.extend(results)
    
    return translated_subtitles

def get_all_file_paths(directory):
    """
    获取目录下所有文件的完整路径
    
    Args:
        directory (str): 要遍历的目录路径
    
    Returns:
        list: 包含所有文件完整路径的列表
    """
    file_paths = []
    
    for root, directories, files in os.walk(directory):
        for filename in files:
            filepath = os.path.join(root, filename)
            file_paths.append(filepath)
    
    return file_paths
def chunk_list(lst, chunk_size):
    """
    通用的列表分块函数
    
    Args:
        lst (list): 输入列表
        chunk_size (int): 每块的大小
    
    Returns:
        list: 分块后的列表
    """
    chunks = []
    for i in range(0, len(lst), chunk_size):
        chunk = lst[i:i+chunk_size]
        chunks.append(''.join(chunk))
    return chunks
def get_groups_of_four(lst):
    
    four = []
    for i in lst:
        if has_english(i):
            four.append(i)
    return four
def merge_sentence(lst):
    merged = []
    temp = ''
    
    for i in lst:
        if temp == '':
            temp = i
        else:
            # 如果当前temp以句号结尾，说明一个句子结束
            if temp.endswith('.'):
                merged.append(temp)
                temp = i  # 开始新的句子，而不是清空
            else:
                temp += i
    
    # 处理最后一个句子
    if temp != '':
        merged.append(temp)
    
    return merged
import re

def has_english(text):
    """
    判断字符串中是否包含英文字母
    """
    pattern = re.compile(r'[a-zA-Z]')
    return bool(pattern.search(text))

In [3]:
paths = get_all_file_paths('data')
if __name__ == '__main__':
    
    paths = get_all_file_paths('data')
    temp = []
    for path in tqdm(paths):
        if path.split('\\')[-1] in [i.split('\\')[ -1 ]for i in  get_all_file_paths('cc')]:
            continue
        with open(path,'r') as f:
            for line in f:
                temp.append(line.strip())
            merged = merge_sentence(get_groups_of_four(temp))
            temp = []
            with open('en/' + path.split('_')[2].strip() + '.json','w', encoding='utf-8') as f:
                json.dump(merged,f,indent=2,ensure_ascii=False)
        
        chunk_merged = chunk_list(merged,50)
        print(f'开始翻译{path.split('_')[2]}')
        ch = batch_translate_subtitles_threaded(chunk_merged,batch_size=1, max_workers= 100)
        print("成功")
        with open('ch/' + path.split('_')[2].strip() + '.json','w', encoding='utf-8') as f:
                json.dump(ch,f,indent=2,ensure_ascii=False)

  0%|          | 0/17 [00:00<?, ?it/s]

开始翻译 Lecture 8


100%|██████████| 17/17 [00:29<00:00,  1.72s/it]

成功
